# Конкурентный NPS: пять проверок на синтетических анкетах

Условные банки A и B. В примере склонность оценить продукт и сами оценки зависят от сегмента. Все ответы сгенерированы. Численные результаты демонстрируют методику и не описывают клиентов какого-либо банка.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression, LinearRegression

rng = np.random.default_rng(42)
n = 10000
segment = rng.binomial(1, 0.45, size=n)
latent = rng.normal(size=n)
scores = {}
for bank, offset, prob in [("A", 0.2, 0.35 + 0.4 * segment), ("B", 0.0, 0.7 - 0.3 * segment)]:
    response = rng.random(n) < prob
    rating = np.clip(np.rint(7 + offset + 1.3 * segment + latent + rng.normal(0, 1.7, n)), 0, 10)
    scores[bank] = np.where(response, np.where(rating >= 9, 100, np.where(rating <= 6, -100, 0)), np.nan)
frame = pd.DataFrame(scores)
print("Ответили по банкам:", frame.count().to_dict())

Ответили по банкам: {'A': 5255, 'B': 5647}


## Точность NPS и состав аудиторий

NPS — средний вклад респондента в пунктах. Для демонстрации считаем процентильный интервал по бутстрэпу респондентов и стандартизованную разность долей для единственного наблюдаемого признака.

In [2]:
valid = frame.notna()
for bank in frame:
    y = frame[bank].dropna().to_numpy()
    boots = [rng.choice(y, len(y), replace=True).mean() for _ in range(1000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"{bank}: NPS {y.mean():.1f}, 95% ДИ [{lo:.1f}, {hi:.1f}]")
pa, pb = [segment[valid[bank]].mean() for bank in frame]
smd = (pa - pb) / np.sqrt((pa * (1-pa) + pb * (1-pb)) / 2)
print(f"Доля сегмента 1: A {pa:.1%}, B {pb:.1%}; SMD {smd:.2f}")

A: NPS 16.7, 95% ДИ [14.6, 18.7]
B: NPS -3.1, 95% ДИ [-5.1, -1.1]
Доля сегмента 1: A 63.6%, B 32.2%; SMD 0.66


## Мощность и состав пересечения

Сравниваем банки на одних респондентах. Затем сопоставляем тех, кто оценил оба банка, с остальными респондентами каждого банка. Эти сравнения отвечают на разные вопросы.

In [3]:
both = valid.all(axis=1)
diff = (frame.A - frame.B)[both]
z = norm.ppf(0.975) + norm.ppf(0.80)
print(f"Парных ответов {len(diff)}, разница {diff.mean():.1f} п., "
      f"минимальный обнаруживаемый эффект {z * diff.std() / np.sqrt(len(diff)):.1f} п.")
for bank in frame:
    other = valid[bank] & ~both
    print(f"{bank}: доля сегмента 1 в пересечении и среди остальных "
          f"{segment[both].mean():.1%} / {segment[other].mean():.1%}; "
          f"NPS {frame.loc[both, bank].mean():.1f} / {frame.loc[other, bank].mean():.1f}")
for mde in (5, 10):
    print(f"Плановый пример: SD=100, разница {mde} п., нужно {int(np.ceil((z*100/mde)**2))} парных ответов.")

Парных ответов 2653, разница 6.7 п., минимальный обнаруживаемый эффект 5.2 п.
A: доля сегмента 1 в пересечении и среди остальных 51.0% / 76.5%; NPS 11.8 / 21.6
B: доля сегмента 1 в пересечении и среди остальных 51.0% / 15.5%; NPS 5.1 / -10.4
Плановый пример: SD=100, разница 5 п., нужно 3140 парных ответов.
Плановый пример: SD=100, разница 10 п., нужно 785 парных ответов.


## NPS при общем составе опрошенных

Для каждого банка оцениваем вероятность наличия ответа по сегменту и средний отклик внутри сегмента на других фолдах. Перенос на всех опрошенных опирается на перекрытие и допущение о сопоставимости оценок при одинаковом сегменте; в реальном опросе набор признаков шире.

In [4]:
X = segment[:, None]
for bank in frame:
    y = frame[bank].to_numpy()
    observed = np.isfinite(y)
    p, mu = np.empty(n), np.empty(n)
    for train, test in KFold(5, shuffle=True, random_state=42).split(X):
        answered = train[observed[train]]
        p[test] = LogisticRegression().fit(X[train], observed[train]).predict_proba(X[test])[:, 1]
        mu[test] = LinearRegression().fit(X[answered], y[answered]).predict(X[test])
    w = 1 / p[observed]
    ipw = np.average(y[observed], weights=w)
    pseudo = mu.copy()
    pseudo[observed] += w * (y[observed] - mu[observed])
    est = pseudo.mean()
    half = norm.ppf(.975) * pseudo.std(ddof=1) / np.sqrt(n)
    ess = w.sum()**2 / (w @ w)
    print(f"{bank}: IPW {ipw:.1f}, AIPW {est:.1f} [{est-half:.1f}, {est+half:.1f}], ESS {ess:.0f}")

A: IPW 7.7, AIPW 7.7 [5.5, 10.0], ESS 4556
B: IPW 2.7, AIPW 2.7 [0.6, 4.7], ESS 5269
